In [1]:
import operator
!pip install langgraph langchain-core
!pip install -q -U langgraph langchain-google-genai pydantic openpyxl



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Gemini Key

# Model Define

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key="YOUR_API_KEY"
)

## Define Output

In [3]:
from typing import TypedDict, List, Dict, Any, Optional, Annotated
from pydantic import BaseModel, Field

class RequirementAnalaysis(BaseModel):
  feature: str = Field(description ="The main feature Being Tested")
  actor: str = Field(description ="The User or System interacting with the feature")
  preconditions: List[str] = Field(description ="Conditions That Must Be True Before Testing")
  inputs: List[str] = Field(description ="inputs required to use the feature")
  buisness_rules: List[str] = Field(description ="Buisness rule that must be followed")
  expected_behaviour: str = Field(description ="What system should do when the feature works correctly")
  testable_condition: List[str] = Field(description ="Conditions that Should Be tested")



In [4]:
class TestCase(BaseModel):
  test_case_id:str = Field(description= "unique ID of the test case")
  scenario_id: str= Field(description= "ID of the test scenario this test belongs to")
  Title: str= Field(description= "Title of the test case")
  preconditions: List[str] = Field(description ="Conditions That Must Be Satisfied Before executing the test")
  test_steps: List[str] = Field(description="Step-by-step actions performed by the tester")
  expected_result: str = Field(description="Expected result after executing the tests step")
  priority: str = Field(description="Data required to execute the test case")
  test_type: str = Field(description ="testing type: Frontend, Backend, API, Database, Security, or Performance")

In [5]:
class TestScenario(BaseModel):
  scenario_id: str = Field(description="Unique Identifier for the Scenario")
  title: str = Field(description="Short Title of the Scenario")
  test_type: str = Field(description="Category of Scenario Such as Frontend, Backend, API, Database, Security, Performance")
  category: str = Field(description="Category such as Positive, Negative, Validation, Boundary, Edge Case, Security, or Usability")
  description: str = Field(description="Description of the Scenario what needs to be tested")
  priority: str = Field(description="Priority Of Scenario: High, Medium, or Low")




# Wrappers

In [6]:
class TestScenarioList(BaseModel):
  scenarios: List[TestScenario] = Field(description=" Complete Relevant List of Test Scenarios")

class TestCaseList(BaseModel):
  test_cases: List[TestCase] = Field(description=" Complete Relevant List of Test Cases")

# Structured Output

In [7]:
structured_requirement_llm = llm.with_structured_output(
    RequirementAnalaysis
)

structured_scenario_llm = llm.with_structured_output(
    TestScenarioList
)
structured_test_case_llm = llm.with_structured_output(
    TestCaseList
)

# Define Graph State ( Connects Everything )

In [8]:

class TestState(TypedDict):
  requirement:str
  application_url:str
  requirement_analysis: RequirementAnalaysis
  scenarios: List[TestScenario]
  test_case: List[TestCase]
  excel_file_path: Optional[str]

# Define Nodes

In [9]:
def analyze_requirements(state: TestState):
  print("Analyzing Requirements")

  requirement = state["requirement"]
  prompt = f"""
  You are a Senior Software Qa Enginner
  Analyze The following software requirements
  your responsibility is to understand the requirementa
  before generating test scnearios or test cases

  Extract:

  1.Main feature
  2.The actor or user
  3.Preconditions
  4.Inputs
  5.Buisness rules
  6.Expected behaviour
  7.Testable condition

  Do Not Gernarate detailed test cases
  Use Less Tokens As You can in output donot waste tokens!
  Requirements:

  {requirement}
  """

  result = structured_requirement_llm.invoke(prompt)
  return {
      "requirement_analysis": result
  }

In [10]:
def generate_scenario(state: TestState):
  print("Generating Scenarios")

  requirement_analysis = state["requirement_analysis"]

  prompt = f"""
You are a Senior Software Qa Enginnner.
Generate a comprehensive list of test scenarios
for the following requirement analysis.

Generate MULTIPLE test scenarios.
  Requirements Analysis:
  Feature:
  {requirement_analysis.feature}
  Actor:
  {requirement_analysis.actor}
  Preconditions:
  {requirement_analysis.preconditions}
  Inputs:
  {requirement_analysis.inputs}
  Buisness Rules:
  {requirement_analysis.buisness_rules}
  Expected Behaviour:
  {requirement_analysis.expected_behaviour}
  Testable Conditions:
  {requirement_analysis.testable_condition}

TESTING TYPES

Identify all relevant testing types for this requirement.
Possible testing types are:
1. Frontend
   Test user interface behavior, form validation,
   user interactions, visual behavior, and client-side validation.
2. Backend
   Test server-side business logic, business rules,
   error handling, and backend processing.
3. API
   Test API requests, responses, status codes,
   request validation, response validation, and authentication.
4. Database
   Test data persistence, data integrity,
   data creation, updates, and deletion.
5. Security
   Test authentication, authorization,
   sensitive data handling, and security-related behavior.
6. Performance
   Test response time, load behavior,
   and performance-related requirements.
Only generate scenarios for testing types that are
actually relevant to the requirement.
Do NOT force every requirement to have scenarios
for every testing type.
Think Step By Step To target the edge cases

SCENARIO_CATEGORIES

For each relevant testing type, consider:

- Positive
- Negative
- Validation
- Boundary
- Edge Case
- Error Handling
- Business Rules
- Security
- Usability

Only use categories that are relevant to the requirement.

IMPORTANT_RULES

1. Generate MULTIPLE test scenarios when multiple
   scenarios are relevant.
2. Cover all important testable conditions.
3. Do not generate duplicate scenarios.
4. Do not generate detailed test steps.
5. Focus on WHAT should be tested, not HOW to test it.
6. Assign each scenario a unique scenario ID.
7. Assign priority as High, Medium, or Low.
8. Each scenario must have a testing type such as:
   Frontend, Backend, API, Database, Integration,
   Security, or Performance.
9.  Do not Generate Unecesaary Scenarios if not required.
10. Do not create irrelevant testing scenarios.
11. Think like an experienced manual QA engineer.
"""


  result = structured_scenario_llm.invoke(prompt)
  return {
      "scenarios": result.scenarios
  }

In [11]:
def generate_test_case(state: TestState):
    print("Generating Test Cases")

    scenarios = state["scenarios"]

    # Pass all scenarios to the prompt at once
    prompt = f"""
    You are a Senior Software QA Engineer.
    Generate comprehensive test cases for ALL of the following scenarios:

    {scenarios}

    Return a single combined list of test cases covering every scenario.
    Point To Be noted Think Before Replying
    """

    response = structured_test_case_llm.invoke(prompt)

    # Assuming structured_test_case_llm is configured with TestCaseList,
    # you return its internal list of test cases to match your state definition.
    return {
        "test_case": response.test_cases if hasattr(response, "test_cases") else response
    }

In [12]:
import pandas as pd


def write_to_excel(state: TestState):
  print("Writing to Excel")

  # 1. Grab test cases from state, or use an empty list if none exist
  test_cases = state["test_case"]

  # 2. Convert Pydantic models to dictionaries (if applicable)
  data = [tc.model_dump() for tc in test_cases]

  # 3. Save directly to Excel using pandas
  file_name = "test_cases.xlsx"
  pd.DataFrame(data).to_excel(file_name, sheet_name="Test Cases", index=False)

  print(f"Excel file '{file_name}' created successfully.")

  return {"excel_file": file_name}

# Building Graph

In [13]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TestState)


# Adding Nodes

In [14]:

graph.add_node("analyze_requirements", analyze_requirements)
graph.add_node("generate_scenario", generate_scenario)
graph.add_node("generate_test_case", generate_test_case)
graph.add_node("write_to_excel", write_to_excel)


# Adding Edge

In [15]:
graph.add_edge(START,"analyze_requirements")
graph.add_edge("analyze_requirements", "generate_scenario")
graph.add_edge("generate_scenario", "generate_test_case")
graph.add_edge("generate_test_case", "write_to_excel")
graph.add_edge("write_to_excel", END)

# Compiling Graph

In [16]:
workflow = graph.compile()

# Visualize Graph

In [17]:
try:
  png_data = workflow.get_graph().draw_mermaid_png()
  with open("graph.png", "wb") as f:
    f.write(png_data)
except Exception as e:
  print("Could Not Genarate PNG")

  print(workflow.get_graph().draw_mermaid())

# Run Graph

In [18]:
result = workflow.invoke({
    "requirement": (
        """1.Added a dropdown in settings to select the state
            2. Added a upload Photo icon so a user can upload"""
    ),
})

Analyzing Requirements
Generating Scenarios
Generating Test Cases
Writing to Excel
Excel file 'test_cases.xlsx' created successfully.


# Print Requirement Analysis

In [19]:
# Get the list of test cases from your result dictionary
test_cases_list = result["test_case"]

for tc in test_cases_list:
    print(f"Test Case ID: {tc.test_case_id}")
    print(f"Scenario ID:  {tc.scenario_id}")
    print(f"Title:        {tc.Title}")
    print(f"Test Type:    {tc.test_type}")
    print(f"Priority:     {tc.priority}")
    print(f"Preconditions:{tc.preconditions}")
    print(f"Test Steps:   {tc.test_steps}")
    print(f"Expected:     {tc.expected_result}")
    print("-" * 50)  # Visual separator between test cases

Test Case ID: TC_USC_FT_01_01
Scenario ID:  USC_FT_01
Title:        Verify state dropdown UI visibility and selection option displays correctly
Test Type:    Frontend
Priority:     High
Preconditions:['User is logged into the application', 'User is navigated to the profile settings page']
Test Steps:   ['Locate the State/Region dropdown selection on the profile page', 'Click on the State dropdown to expand it', 'Verify all valid state options are listed in alphabetical order', "Select a valid state (e.g., 'California') from the list"]
Expected:     The dropdown list opens smoothly, displays all valid states, and shows 'California' as the selected value upon selection.
--------------------------------------------------
Test Case ID: TC_USC_FT_02_01
Scenario ID:  USC_FT_02
Title:        Verify photo upload icon click triggers native file selection dialog
Test Type:    Frontend
Priority:     High
Preconditions:['User is logged in', 'User is on the profile settings edit view']
Test Steps: 

In [20]:
test_scenario = result["scenarios"]

for test_scenario in test_scenario:
  print(test_scenario.test_type)
  print(test_scenario.title)
  # print(test_scenario.description)
  # print(test_scenario.category)
  # print(test_scenario.priority)
  print()

Frontend
Verify State Dropdown UI and Selection

Frontend
Verify Photo Upload Icon Interaction

Frontend
Frontend Validation for Unsupported File Extensions

Backend
Backend Processing of State Update

Backend
Backend Image Processing and Storage

Backend
Error Handling for Corrupted Image File

API
Verify State Update API Endpoint

API
Verify Photo Upload API Request and Response

Database
Database Verification for Updated State

Database
Database Verification for Uploaded Photo URL

Security
Security Validation against Malicious File Uploads

Security
Boundary Check for Maximum File Upload Size

Security
Unauthorized Request Access Check

Performance
Profile Photo Upload Response Time Performance

